In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [19]:
class PositionalEncoding(nn.Module):
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        super().__init__()

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        self.num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(self.num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(self.num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [ ]:
class CrossMultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, encoder_output: torch.Tensor, decoder_input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if encoder_output.shape[-1] != self.d_model:
            raise ValueError(f"Expected encoder input feature dimension to be d_model ({self.d_model}), but got {encoder_input.shape[-1]}")

        if decoder_input.shape[-1] != self.d_model:
            raise ValueError(f"Expected decoder input feature dimension to be d_model ({self.d_model}), but got {decoder_input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = decoder_input.shape[0]
        encoder_num_tokens = encoder_output.shape[1]
        decoder_num_tokens = decoder_input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(decoder_input)
        k = self.w_k(encoder_output)
        v = self.w_v(encoder_output)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, decoder_num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, encoder_num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()

        # Recombining all the heads
        sim3 = sim3.view(batch_size, decoder_num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [21]:
class MaskedMultiHeadAttention(nn.Module):
    """
    Compute Masked Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:

        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Creating a matrix where lower triangle is 1, rest is 0
        mask = torch.tril(torch.ones(num_tokens, num_tokens))
        # Dimension: (num_queries, num_keys)

        # Applying the mask to our attention scores
        sim1 = sim1.masked_fill(mask == 0, float('-inf'))

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [22]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [38]:
class DecoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim 
        self.cross_attention_layer = CrossMultiHeadAttention(self.d_model, self.num_heads)
        self.masked_attention_layer = MaskedMultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.norm3 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.masked_attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----CROSS ATTENTION----
        cross_attn_input = self.norm2(x)
        # Normalizing

        cross_attn_output = self.cross_attention_layer(cross_attn_input, encoder_output)
        # Passing though cross attention layer

        x = x + self.dropout(cross_attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm3(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [39]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
hidden_dim = 16

In [40]:
dummy_block_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_block_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_block_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.2043, 0.7985, 0.8031, 0.5806, 0.8084, 0.3507, 0.8437, 0.1312],
         [0.6576, 0.7222, 0.1340, 0.5219, 0.2872, 0.3948, 0.2536, 0.1998],
         [0.6975, 0.2753, 0.9342, 0.0179, 0.9165, 0.0385, 0.4355, 0.8616]],

        [[0.5569, 0.9617, 0.3859, 0.3886, 0.7834, 0.8324, 0.0193, 0.5106],
         [0.2307, 0.2375, 0.8764, 0.3866, 0.5721, 0.2528, 0.5714, 0.6038],
         [0.4032, 0.3220, 0.1521, 0.0220, 0.7879, 0.3028, 0.8964, 0.7406]]])

In [42]:
decoder = DecoderBlock(d_model, num_heads, hidden_dim)
decoder

DecoderBlock(
  (cross_attention_layer): CrossMultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (masked_attention_layer): MaskedMultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm3): LayerNorm((8,), e

In [44]:
dummy_output = decoder(dummy_block_input, dummy_block_input)
dummy_output

tensor([[[ 0.1001, -0.0641,  1.4363, -0.1088,  1.4345,  0.7011,  1.7524,
           0.7916],
         [ 0.6763,  0.6258,  0.4716,  0.5260,  0.3905,  0.7013,  0.4213,
           0.4455],
         [ 0.5892, -0.2274,  1.2793, -0.1841,  1.3124,  0.0030,  1.5705,
           1.3610]],

        [[ 0.1491,  1.2595, -0.1574,  0.6560,  0.4932,  0.9429,  0.6211,
          -0.1037],
         [-0.2187,  0.0756,  0.9011, -0.1184,  0.7931,  0.2112,  1.3560,
           1.3359],
         [ 0.2355,  0.2669,  0.2236, -0.4810,  0.6715,  0.1002,  1.6691,
           1.0177]]], grad_fn=<AddBackward0>)

In [ ]:
# Now, Stack Multiple Decoders:
class DecoderStack(nn.Module):
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding()
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, hidden_dim) for _ in range(num_blocks)])
        self.norm_1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)
        
    def forward(self, tokens: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor: 
        # Dimension of tokens: (batch_size, num_tokens) 

        embedded = self.embedding(tokens) 
        # Dimension: (batch_size, num_tokens, d_model)
        
        embedded_scaled = embedded * math.sqrt(self.d_model)
        positional_embedding = self.positional_encoding(embedded_scaled)
        x = self.dropout(positional_embedding)

        for layer in self.layers:
            x = layer(x, encoder_output)
        output = self.norm_1(x)

        return output

In [34]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 5
hidden_dim = 16
num_blocks = 6

In [35]:
dummy_stack_input = torch.randint(low=0, high=vocab_size, size=(batch_size, num_tokens))
print(dummy_stack_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_stack_input

torch.Size([2, 3])
(batch_size, num_tokens, embedding_dimension)


tensor([[4, 3, 3],
        [0, 3, 2]])

In [36]:
decoder_stack = DecoderStack(vocab_size, d_model, num_heads, hidden_dim, num_blocks, dummy_block_input)
decoder_stack

DecoderStack(
  (embedding): Embedding(5, 8)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-5): 6 x DecoderBlock(
      (cross_attention_layer): CrossMultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (masked_attention_layer): MaskedMultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (

In [37]:
decoder_stack_output = decoder_stack(dummy_stack_input)
print(decoder_stack_output.shape)
decoder_stack_output

torch.Size([2, 3, 8])


tensor([[[-0.6316,  0.7792, -1.4714, -0.6110,  1.7466,  0.8618, -0.8034,
           0.1298],
         [ 0.1821,  0.4944, -1.0369,  0.1890, -0.6352,  2.2398, -1.0030,
          -0.4303],
         [ 0.0370,  0.1392, -0.5741,  0.0837, -0.5406,  2.4354, -1.0819,
          -0.4987]],

        [[-0.6281, -0.0326, -0.2308,  1.9662, -0.9887, -0.5831,  1.3078,
          -0.8108],
         [ 0.2869,  0.5237, -1.1407,  0.4410, -0.3686,  2.0659, -1.1377,
          -0.6705],
         [ 0.2976,  1.0191, -1.4056, -0.4236, -0.8268,  1.9479, -0.1392,
          -0.4693]]], grad_fn=<NativeLayerNormBackward0>)